In [1]:
import torch
import torch.nn as nn

In [2]:
batch_size = 4

sequence_length = 10

input_size = 8

hidden_size = 32

num_layers = 3

num_classes = 5

x = torch.randn(
    batch_size,
    sequence_length,
    input_size
)

print("Input Shape :", x.shape)

Input Shape : torch.Size([4, 10, 8])


In [3]:
class ManyToManyRNN(nn.Module):

    def __init__(self):

        super().__init__()

        self.rnn = nn.RNN(

            input_size=input_size,

            hidden_size=hidden_size,

            num_layers=num_layers,

            batch_first=True

        )

        self.fc = nn.Linear(
            hidden_size,
            num_classes
        )

    def forward(self, x):

        output, hidden = self.rnn(x)

        print("RNN Output Shape :", output.shape)

        print("Hidden Shape :", hidden.shape)

        output = self.fc(output)

        return output

In [4]:
model = ManyToManyRNN()

output = model(x)

print("Final Output Shape :", output.shape)

RNN Output Shape : torch.Size([4, 10, 32])
Hidden Shape : torch.Size([3, 4, 32])
Final Output Shape : torch.Size([4, 10, 5])


In [5]:
class ManyToOneRNN(nn.Module):

    def __init__(self):

        super().__init__()

        self.rnn = nn.RNN(

            input_size=input_size,

            hidden_size=hidden_size,

            num_layers=num_layers,

            batch_first=True

        )

        self.fc = nn.Linear(
            hidden_size,
            num_classes
        )

    def forward(self, x):

        output, hidden = self.rnn(x)

        print("RNN Output Shape :", output.shape)

        print("Hidden Shape :", hidden.shape)

        # Last Time Step
        output = output[:, -1, :]

        print("Last Time Step :", output.shape)

        output = self.fc(output)

        return output

In [6]:
model = ManyToOneRNN()

output = model(x)

print("Final Output Shape :", output.shape)

RNN Output Shape : torch.Size([4, 10, 32])
Hidden Shape : torch.Size([3, 4, 32])
Last Time Step : torch.Size([4, 32])
Final Output Shape : torch.Size([4, 5])


In [9]:
batch_size = 4

input_size = 8

num_classes = 5

x = torch.randn(batch_size, input_size)

print("Input Shape :", x.shape)

class OneToOneNN(nn.Module):

    def __init__(self):

        super().__init__()

        self.network = nn.Sequential(

            nn.Linear(input_size,64),
            nn.ReLU(),

            nn.Linear(64,32),
            nn.ReLU(),

            nn.Linear(32,16),
            nn.ReLU(),

            nn.Linear(16,num_classes)

        )

    def forward(self,x):

        return self.network(x)

model = OneToOneNN()

output = model(x)

print("Output Shape :", output.shape)

Input Shape : torch.Size([4, 8])
Output Shape : torch.Size([4, 5])


In [12]:
batch_size = 4

input_size = 8

sequence_length = 10

hidden_size = 32

num_layers = 3

num_classes = 5

x = torch.randn(batch_size, input_size)

print("Input Shape :", x.shape)

class OneToManyRNN(nn.Module):

    def __init__(self):

        super().__init__()

        self.fc = nn.Linear(
            input_size,
            hidden_size
        )

        self.rnn = nn.RNN(

            input_size=hidden_size,

            hidden_size=hidden_size,

            num_layers=num_layers,

            batch_first=True

        )

        self.output = nn.Linear(
            hidden_size,
            num_classes
        )

    def forward(self,x):

        # Convert single input to hidden vector
        x = self.fc(x)

        print("After FC :", x.shape)

        # Repeat the same vector for every time step
        x = x.unsqueeze(1)

        x = x.repeat(
            1,
            sequence_length,
            1
        )

        print("Repeated Input :", x.shape)

        output, hidden = self.rnn(x)

        print("RNN Output :", output.shape)

        print("Hidden Shape :", hidden.shape)

        output = self.output(output)

        return output

model = OneToManyRNN()

output = model(x)

print("Final Output Shape :", output.shape)

Input Shape : torch.Size([4, 8])
After FC : torch.Size([4, 32])
Repeated Input : torch.Size([4, 10, 32])
RNN Output : torch.Size([4, 10, 32])
Hidden Shape : torch.Size([3, 4, 32])
Final Output Shape : torch.Size([4, 10, 5])


In [13]:
batch_size = 4

Tx = 10

Ty = 6

input_size = 8

hidden_size = 32

num_layers = 3

num_classes = 5

x = torch.randn(
    batch_size,
    Tx,
    input_size
)

print("Input Shape :", x.shape)

Input Shape : torch.Size([4, 10, 8])


In [14]:
class ManyToManyDifferentRNN(nn.Module):

    def __init__(self):

        super().__init__()

        self.encoder = nn.RNN(

            input_size=input_size,

            hidden_size=hidden_size,

            num_layers=num_layers,

            batch_first=True

        )

        self.decoder = nn.RNN(

            input_size=hidden_size,

            hidden_size=hidden_size,

            num_layers=num_layers,

            batch_first=True

        )

        self.fc = nn.Linear(

            hidden_size,

            num_classes

        )

    def forward(self, x):

        # -----------------------
        # Encoder
        # -----------------------

        encoder_output, hidden = self.encoder(x)

        print("Encoder Output :", encoder_output.shape)

        print("Encoder Hidden :", hidden.shape)

        # -----------------------
        # Decoder Input
        # -----------------------

        decoder_input = torch.zeros(

            x.size(0),

            Ty,

            hidden_size

        )

        print("Decoder Input :", decoder_input.shape)

        # -----------------------
        # Decoder
        # -----------------------

        decoder_output, hidden = self.decoder(

            decoder_input,

            hidden

        )

        print("Decoder Output :", decoder_output.shape)

        output = self.fc(decoder_output)

        return output

In [15]:
model = ManyToManyDifferentRNN()

output = model(x)

print("Final Output Shape :", output.shape)

Encoder Output : torch.Size([4, 10, 32])
Encoder Hidden : torch.Size([3, 4, 32])
Decoder Input : torch.Size([4, 6, 32])
Decoder Output : torch.Size([4, 6, 32])
Final Output Shape : torch.Size([4, 6, 5])
